# 03 Facets Reviews

Computes review-side diversity facets M0-M5 over `condition × text_version × field`. Reviews are analyzed as paired, per-target-proposal exact-n panels using the row-index cache written by `4b`. M6 is not applicable to reviews.

In [1]:
CONFIG = dict(
    conditions=["baseline", "one_at_a_time", "persona"],
    text_versions=["rephrased", "original"],
    fields=["whole", "strengths", "weakness"],
    models=["claude", "gemini", "gpt"],
    n_human=23,
    seed=42,
    B_perm=10_000,
    B_sub=1_000,
)

RUN_OT = True
WRITE_FIGURES = True

## Load

Load review artifacts, assert row-order contracts, and convert exact-n row-index panels into the UID-combination shape expected by the review facet helper.

In [2]:
import json
import pickle
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import diversity_inference as di
from compare_review_diversity import load_pickle, load_review_analysis_inputs

if not RUN_OT:
    di.wasserstein_ot = lambda X, Y: np.nan

TEST_COLUMNS = [
    "condition", "task", "text_version", "field", "comparison", "facet", "metric", "is_primary", "param",
    "human_value", "ai_value", "effect_size", "effect_type", "ci_lo", "ci_hi", "inference", "stat",
    "p_raw", "p_fdr", "n_human", "n_ai", "n_perm_or_sub", "parity_ref", "notes",
]
PRIMARY = {
    ("spread", "mean_pairwise", ""),
    ("richness", "vendi", "q=1"),
    ("coverage", "coverage_geometric", "k=panel_adaptive"),
    ("dimensionality", "participation_ratio", ""),
    ("evenness", "ripley_excess", "r=panel_q01_q50"),
    ("displacement", "mmd2", ""),
}

def _load_json(path):
    return json.loads(Path(path).read_text())

def _standardize_tests(df):
    out = df.copy()
    if "text_branch" in out.columns:
        out = out.rename(columns={"text_branch": "text_version"})
    if "n_perm_or_boot" in out.columns:
        out = out.rename(columns={"n_perm_or_boot": "n_perm_or_sub"})
    for col in TEST_COLUMNS:
        if col not in out.columns:
            out[col] = np.nan if col not in {"field", "param", "notes"} else ""
    out["is_primary"] = out.apply(lambda r: (r["facet"], r["metric"], r["param"]) in PRIMARY, axis=1)
    out.loc[out["facet"].ne("coverage"), "parity_ref"] = 1.0
    out.loc[out["metric"].eq("coverage_geometric"), "parity_ref"] = out.loc[out["metric"].eq("coverage_geometric"), "human_value"]
    return out[TEST_COLUMNS]

def _standardize_gradient(df):
    out = df.copy()
    if "text_branch" in out.columns:
        out = out.rename(columns={"text_branch": "text_version"})
    for col in ["condition", "task", "text_version", "field", "facet", "metric", "param", "order", "JT", "p_raw", "p_fdr", "direction_ok", "notes"]:
        if col not in out.columns:
            out[col] = np.nan if col in {"JT", "p_raw", "p_fdr"} else ""
    return out[["condition", "task", "text_version", "field", "facet", "metric", "param", "order", "JT", "p_raw", "p_fdr", "direction_ok", "notes"]]

def _standardize_curves(df):
    out = df.copy()
    if "text_branch" in out.columns:
        out = out.rename(columns={"text_branch": "text_version"})
    for col in ["condition", "task", "text_version", "field", "group", "facet", "metric", "x", "y", "y_lo", "y_hi"]:
        if col not in out.columns:
            out[col] = np.nan if col in {"x", "y", "y_lo", "y_hi"} else ""
    for numeric_col in ["x", "y", "y_lo", "y_hi"]:
        out[numeric_col] = pd.to_numeric(out[numeric_col], errors="coerce")
    return out[["condition", "task", "text_version", "field", "group", "facet", "metric", "x", "y", "y_lo", "y_hi"]]

def _standardize_paired(df):
    out = df.copy()
    if "text_branch" in out.columns:
        out = out.rename(columns={"text_branch": "text_version"})
    for col in ["condition", "text_version", "field", "comparison", "facet", "metric", "param", "target_proposal_uid", "target_cohort", "n_human_reviews", "human_value", "ai_value", "paired_diff"]:
        if col not in out.columns:
            out[col] = np.nan if col in {"n_human_reviews", "human_value", "ai_value", "paired_diff"} else ""
    return out[["condition", "text_version", "field", "comparison", "facet", "metric", "param", "target_proposal_uid", "target_cohort", "n_human_reviews", "human_value", "ai_value", "paired_diff"]]

def _assert_review_contract(condition, text_version):
    prep_dir = PROJECT_ROOT / "data" / "prepared" / condition / "reviews" / text_version
    manifest = _load_json(prep_dir / "prepare_manifest.json")
    master = pd.read_csv(prep_dir / "review_master.csv")
    assert manifest["embeddings_l2_normalized"] is True, f"run modified 4b first: {condition}/{text_version}"
    assert manifest["review_uid_order"] == master["review_uid"].astype(str).tolist(), "row order drift in review master"
    assert "source_family" in master.columns, "review model family column missing: expected source_family"
    return prep_dir, manifest, master

def _panels_to_combination_cache(master, panels):
    cache = {"all_ai": {}, "claude": {}, "gemini": {}, "gpt": {}}
    review_uids = master["review_uid"].astype(str).tolist()
    for uid, payload in panels.items():
        human_uids = [review_uids[int(i)] for i in payload["human_idx"]]
        cache["all_ai"][uid] = {
            "eligible": True,
            "human_review_uids": human_uids,
            "ai_panel_combinations": [[review_uids[int(i)] for i in combo] for combo in payload["pooled"]],
        }
        for model in ["claude", "gemini", "gpt"]:
            cache[model][uid] = {
                "eligible": True,
                "human_review_uids": human_uids,
                "ai_panel_combinations": [[review_uids[int(i)] for i in combo] for combo in payload["per_model"].get(model, [])],
            }
    return cache

def _analysis_for_field(condition, text_version, field, manifest):
    analysis = load_review_analysis_inputs(PROJECT_ROOT, condition, text_version=text_version)
    if field == "whole":
        return analysis
    field_key = f"{field}_embedding_path"
    if field_key not in manifest:
        return None
    bundle = load_pickle(Path(manifest[field_key]))
    return replace(analysis, review_embeddings=bundle)

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/eveyhuang/Documents/NICO/human-AI-proposal


## Panels

Use `review_panels_exact_n.pkl` from `4b`; indices are positional and checked against `review_uid_order` before conversion.

In [3]:
all_tests = []
all_gradients = []
all_curves = []
all_paired = []

for condition in CONFIG["conditions"]:
    for text_version in CONFIG["text_versions"]:
        prep_dir, manifest, master = _assert_review_contract(condition, text_version)
        with open(manifest["review_panels_exact_n_file"], "rb") as fh:
            panels = pickle.load(fh)
        combination_cache = _panels_to_combination_cache(master, panels)
        for field in CONFIG["fields"]:
            if field not in manifest.get("fields_available", ["whole"]):
                print(f"Skipping unavailable field: {condition}/{text_version}/{field}")
                continue
            print(f"\n=== Reviews: {condition}/{text_version}/{field} ===")
            analysis = _analysis_for_field(condition, text_version, field, manifest)
            if analysis is None:
                continue
            tests, gradients, curves, paired = di.build_review_facet_outputs(
                analysis,
                combination_cache,
                text_branch=text_version,
                field=field,
                n_boot=CONFIG["B_sub"],
                seed=CONFIG["seed"],
            )
            all_tests.append(_standardize_tests(tests))
            all_gradients.append(_standardize_gradient(gradients))
            all_curves.append(_standardize_curves(curves))
            all_paired.append(_standardize_paired(paired))

review_tests_df = pd.concat(all_tests, ignore_index=True) if all_tests else pd.DataFrame(columns=TEST_COLUMNS)
review_gradient_df = pd.concat(all_gradients, ignore_index=True) if all_gradients else pd.DataFrame()
review_curves_df = pd.concat(all_curves, ignore_index=True) if all_curves else pd.DataFrame()
review_paired_df = pd.concat(all_paired, ignore_index=True) if all_paired else pd.DataFrame()
review_tests_df.head()


=== Reviews: baseline/rephrased/whole ===

=== Reviews: baseline/rephrased/strengths ===

=== Reviews: baseline/rephrased/weakness ===

=== Reviews: baseline/original/whole ===
Skipping unavailable field: baseline/original/strengths
Skipping unavailable field: baseline/original/weakness

=== Reviews: one_at_a_time/rephrased/whole ===

=== Reviews: one_at_a_time/rephrased/strengths ===

=== Reviews: one_at_a_time/rephrased/weakness ===

=== Reviews: one_at_a_time/original/whole ===
Skipping unavailable field: one_at_a_time/original/strengths
Skipping unavailable field: one_at_a_time/original/weakness

=== Reviews: persona/rephrased/whole ===

=== Reviews: persona/rephrased/strengths ===

=== Reviews: persona/rephrased/weakness ===

=== Reviews: persona/original/whole ===
Skipping unavailable field: persona/original/strengths
Skipping unavailable field: persona/original/weakness


,condition,task,text_version,field,comparison,facet,metric,is_primary,param,human_value,...,ci_hi,inference,stat,p_raw,p_fdr,n_human,n_ai,n_perm_or_sub,parity_ref,notes
0,baseline,reviews,rephrased,whole,human_vs_pooled_ai,richness,vendi,True,q=1,1.183899,...,0.005877,paired_wilcoxon,6.0,0.000003,0.000006,3.695652,3.695652,23.0,1.0,paired across target proposals; AI value is me...
1,baseline,reviews,rephrased,whole,human_vs_pooled_ai,dimensionality,participation_ratio,True,,2.515631,...,0.390695,paired_wilcoxon,15.0,0.000295,0.000433,3.695652,3.695652,23.0,1.0,paired across target proposals; AI value is me...
2,baseline,reviews,rephrased,whole,human_vs_pooled_ai,dimensionality,effective_rank,False,,2.598130,...,0.222455,paired_wilcoxon,13.0,0.000229,0.000354,3.695652,3.695652,23.0,1.0,paired across target proposals; AI value is me...
3,baseline,reviews,rephrased,whole,human_vs_pooled_ai,coverage,coverage_geometric,True,k=panel_adaptive,1.000000,...,0.636630,paired_wilcoxon,0.0,0.000027,0.000046,3.695652,3.695652,23.0,1.0,paired across target proposals; AI value is me...
4,baseline,reviews,rephrased,whole,human_vs_pooled_ai,evenness,vendi_slope,False,q=0..2,-0.613704,...,0.001110,paired_wilcoxon,6.0,0.000003,0.000006,3.695652,3.695652,23.0,1.0,paired across target proposals; AI value is me...


## Export

Write paired review facet outputs per `{condition}/reviews/{text_version}` and cross-condition copies.

In [4]:
def _write_curves(path, df):
    out = df.copy()
    for numeric_col in ["x", "y", "y_lo", "y_hi"]:
        if numeric_col in out.columns:
            out[numeric_col] = pd.to_numeric(out[numeric_col], errors="coerce")
    try:
        out.to_parquet(path, index=False)
    except Exception as exc:
        raise RuntimeError(f"Could not write required parquet {path}: {exc}") from exc

def _write_cell_outputs(condition, text_version):
    tables_dir = PROJECT_ROOT / "results" / "tables" / condition / "reviews" / text_version
    figures_dir = PROJECT_ROOT / "results" / "figures" / condition / "reviews" / text_version
    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)
    tests = review_tests_df[(review_tests_df.condition == condition) & (review_tests_df.text_version == text_version)].copy()
    gradients = review_gradient_df[(review_gradient_df.condition == condition) & (review_gradient_df.text_version == text_version)].copy()
    curves = review_curves_df[(review_curves_df.condition == condition) & (review_curves_df.text_version == text_version)].copy()
    paired = review_paired_df[(review_paired_df.condition == condition) & (review_paired_df.text_version == text_version)].copy()
    tests.to_csv(tables_dir / "facet_diversity_tests.csv", index=False)
    gradients.to_csv(tables_dir / "facet_diversity_gradient.csv", index=False)
    _write_curves(tables_dir / "facet_diversity_curves.parquet", curves)
    paired.to_csv(tables_dir / "facet_review_paired_long.csv", index=False)
    return tables_dir, figures_dir

written = []
for condition in CONFIG["conditions"]:
    for text_version in CONFIG["text_versions"]:
        written.append((condition, text_version, *_write_cell_outputs(condition, text_version)))

for text_version in CONFIG["text_versions"]:
    cross_dir = PROJECT_ROOT / "results" / "tables" / "cross_condition" / "reviews" / text_version
    cross_dir.mkdir(parents=True, exist_ok=True)
    review_tests_df[review_tests_df.text_version.eq(text_version)].to_csv(cross_dir / "facet_diversity_tests.csv", index=False)
    review_gradient_df[review_gradient_df.text_version.eq(text_version)].to_csv(cross_dir / "facet_diversity_gradient.csv", index=False)
    _write_curves(cross_dir / "facet_diversity_curves.parquet", review_curves_df[review_curves_df.text_version.eq(text_version)])
    review_paired_df[review_paired_df.text_version.eq(text_version)].to_csv(cross_dir / "facet_review_paired_long.csv", index=False)

pd.DataFrame(written, columns=["condition", "text_version", "tables_dir", "figures_dir"])

,condition,text_version,tables_dir,figures_dir
0,baseline,rephrased,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
1,baseline,original,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
2,one_at_a_time,rephrased,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
3,one_at_a_time,original,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
4,persona,rephrased,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
5,persona,original,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...


## Figures

Regenerate required review figures from the tidy tables and paired-detail CSVs.

In [6]:
REQUIRED_REVIEW_FIGURES = [
    ("spread_mean_pairwise_box", "Spread — mean_pairwise", "paired_box", "spread", "mean_pairwise"),
    ("spread_mean_pairwise_ridge", "Spread — mean_pairwise", "paired_hist", "spread", "mean_pairwise"),
    ("spread_mean_pairwise_effect", "Spread — mean_pairwise", "effect", "spread", "mean_pairwise"),
    ("spread_convergent_box", "Spread — convergent metrics", "convergent_box", "spread", None),
    ("richness_vendi_profile", "Richness — Vendi profile", "paired_profile", "richness", "vendi"),
    ("richness_vendi_scree", "Richness — Vendi scree", "curve_required", "richness", "kernel_eigen_scree"),
    ("richness_vendi_box", "Richness — Vendi VS1", "paired_box", "richness", "vendi"),
    ("richness_vendi_effect", "Richness — Vendi VS1", "effect", "richness", "vendi"),
    ("evenness_ripley_excess_envelope", "Evenness — ripley_excess", "curve_required", "evenness", "ripley_K"),
    ("evenness_g_function_cdf", "Evenness — G-function", "curve_required", "evenness", "g_function"),
    ("evenness_nn_similarity_hist", "Evenness — NN similarities", "paired_hist", "spread", "nn_isolation"),
    ("evenness_vendi_slope_box", "Evenness — vendi_slope", "paired_box", "evenness", "vendi_slope"),
    ("dimensionality_participation_ratio_scree", "Dimensionality — participation_ratio", "curve_required", "dimensionality", "participation_ratio_scree"),
    ("dimensionality_participation_ratio_box", "Dimensionality — participation_ratio", "paired_box", "dimensionality", "participation_ratio"),
    ("dimensionality_participation_ratio_effect", "Dimensionality — participation_ratio", "effect", "dimensionality", "participation_ratio"),
    ("coverage_geometric_scatter", "Coverage — geometric", "coverage_scatter", "coverage", "coverage_geometric"),
    ("coverage_geometric_box", "Coverage — geometric", "paired_box", "coverage", "coverage_geometric"),
    ("coverage_geometric_effect", "Coverage — geometric", "effect", "coverage", "coverage_geometric"),
    ("displacement_mmd2_bar", "Displacement — MMD2", "effect", "displacement", "mmd2"),
    ("review_space_umap", "Review-space UMAP illustration", "umap", None, None),
]

GROUP_COLORS = {
    "human_vs_claude": "#4A90E2",
    "human_vs_gemini": "#7B68EE",
    "human_vs_gpt": "#3CB371",
    "human_vs_pooled_ai": "#4A90E2",
}

def _save_fig(fig, path_base):
    path_base.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path_base.with_suffix(".png"), dpi=300, bbox_inches="tight")
    # fig.savefig(path_base.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)

def _whole(df):
    return df[df["field"].eq("whole")].copy() if "field" in df.columns else df.copy()

def _metric_rows(df, facet, metric):
    out = _whole(df)
    if facet is not None:
        out = out[out["facet"].eq(facet)]
    if metric is not None:
        out = out[out["metric"].eq(metric)]
    return out.copy()

def _empty_panel(out_base, title, message):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.text(0.5, 0.5, message, ha="center", va="center", wrap=True)
    ax.set_title(title)
    ax.set_axis_off()
    _save_fig(fig, out_base)

def _plot_paired_box(paired, out_base, title, facet, metric):
    sub = _metric_rows(paired, facet, metric)
    if sub.empty:
        return _empty_panel(out_base, title, f"No paired rows for {facet}/{metric}.")
    fig, ax = plt.subplots(figsize=(8, 4.8))
    order = [c for c in ["human_vs_claude", "human_vs_gemini", "human_vs_gpt", "human_vs_pooled_ai"] if c in set(sub["comparison"])]
    data = [sub.loc[sub["comparison"].eq(c), "paired_diff"].dropna().to_numpy() for c in order]
    ax.boxplot(data, tick_labels=[c.replace("human_vs_", "") for c in order], patch_artist=True)
    rng = np.random.default_rng(42)
    for x, vals in enumerate(data, start=1):
        ax.scatter(x + rng.uniform(-0.08, 0.08, size=len(vals)), vals, s=14, alpha=0.45, color="#404040")
    ax.axhline(0.0, color="#404040", linestyle="--", linewidth=1)
    ax.set_ylabel("paired Human - AI value")
    ax.set_title(title + "\npoints = 23 paired target proposals")
    fig.tight_layout()
    _save_fig(fig, out_base)

def _plot_paired_hist(paired, out_base, title, facet, metric):
    sub = _metric_rows(paired, facet, metric)
    if sub.empty:
        return _empty_panel(out_base, title, f"No paired rows for {facet}/{metric}.")
    fig, ax = plt.subplots(figsize=(8, 4.8))
    for comp, grp in sub.groupby("comparison"):
        ax.hist(grp["paired_diff"].dropna(), bins=12, alpha=0.35, label=comp.replace("human_vs_", ""), color=GROUP_COLORS.get(comp))
    ax.axvline(0.0, color="#404040", linestyle="--", linewidth=1)
    ax.set_xlabel("paired Human - AI value")
    ax.set_ylabel("proposal count")
    ax.set_title(title + "\npoints = paired target-proposal values")
    ax.legend(fontsize=8)
    fig.tight_layout()
    _save_fig(fig, out_base)

def _plot_effect(tests, out_base, title, facet, metric):
    sub = _metric_rows(tests, facet, metric)
    if sub.empty:
        return _empty_panel(out_base, title, f"No test rows for {facet}/{metric}.")
    value_col = "effect_size" if facet in {"displacement"} else "ai_value"
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    plot = sub.groupby("comparison", as_index=False)[value_col].mean().sort_values(value_col)
    colors = [GROUP_COLORS.get(c, "#808080") for c in plot["comparison"]]
    ax.barh(plot["comparison"].str.replace("human_vs_", "", regex=False), plot[value_col], color=colors)
    ax.set_xlabel("MMD2 distance" if facet == "displacement" else "AI value (paired mean)")
    ax.set_title(title)
    fig.tight_layout()
    _save_fig(fig, out_base)

def _plot_convergent_box(paired, out_base, title):
    sub = _whole(paired)
    sub = sub[sub["facet"].eq("spread") & sub["metric"].isin(["centroid_loo", "mst_dispersion", "sparseness", "nn_isolation", "spherical_variance"])]
    if sub.empty:
        return _empty_panel(out_base, title, "No convergent spread rows.")
    fig, ax = plt.subplots(figsize=(10, 5))
    labels, data = [], []
    for metric, grp in sub.groupby("metric"):
        labels.append(metric)
        data.append(grp["paired_diff"].dropna().to_numpy())
    ax.boxplot(data, tick_labels=labels, patch_artist=True)
    ax.axhline(0.0, color="#404040", linestyle="--", linewidth=1)
    ax.set_ylabel("paired Human - AI value")
    ax.set_title(title + "\npoints = paired target-proposal values")
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    _save_fig(fig, out_base)

def _plot_paired_profile(paired, out_base, title, facet, metric):
    sub = _metric_rows(paired, facet, metric)
    if sub.empty:
        return _empty_panel(out_base, title, f"No paired rows for {facet}/{metric}.")
    fig, ax = plt.subplots(figsize=(8, 5))
    comps = [c for c in ["human_vs_claude", "human_vs_gemini", "human_vs_gpt", "human_vs_pooled_ai"] if c in set(sub["comparison"])]
    for x, comp in enumerate(comps):
        grp = sub[sub["comparison"].eq(comp)]
        human = grp["human_value"].to_numpy()
        ai = grp["ai_value"].to_numpy()
        ax.scatter(np.full_like(human, x - 0.12, dtype=float), human, s=14, color="#DC143C", alpha=0.45)
        ax.scatter(np.full_like(ai, x + 0.12, dtype=float), ai, s=14, color=GROUP_COLORS.get(comp), alpha=0.45)
        ax.plot([x - 0.12, x + 0.12], [np.nanmean(human), np.nanmean(ai)], color="#404040", linewidth=2)
    ax.set_xticks(range(len(comps)), [c.replace("human_vs_", "") for c in comps], rotation=20)
    ylabel = {
        ("richness", "vendi"): "Vendi VS1",
        ("coverage", "coverage_geometric"): "geometric coverage",
        ("dimensionality", "participation_ratio"): "participation ratio",
        ("evenness", "ripley_excess"): "Ripley excess area",
        ("spread", "mean_pairwise"): "mean pairwise distance",
    }.get((facet, metric), metric or "value")
    ax.set_ylabel(ylabel)
    ax.set_title(title + "\nred = Human panel; colored = AI exact-n panel mean")
    fig.tight_layout()
    _save_fig(fig, out_base)

def _plot_curve_required(curves, out_base, title, facet, metric):
    sub = curves.copy()
    if not sub.empty:
        sub = sub[(sub["facet"].eq(facet)) & (sub["metric"].eq(metric)) & (sub.get("field", "whole").eq("whole") if "field" in sub.columns else True)]
    if sub.empty:
        return _empty_panel(
            out_base,
            title,
            f"Curve rows for {facet}/{metric} are missing from facet_diversity_curves.parquet.\n"
            "This figure should not be interpreted until the review compute section emits the required curve data.",
        )
    axis_labels = {
        "kernel_eigen_scree": ("eigen-index", "normalized kernel eigenvalue"),
        "participation_ratio_scree": ("principal component index", "cumulative variance share"),
        "ripley_K": ("cosine-distance radius", "Ripley K(r)"),
        "g_function": ("cosine-distance radius", "G(r)"),
    }
    fig, ax = plt.subplots(figsize=(8, 4.8))
    envelope_drawn = False
    for group, grp in sub.groupby("group"):
        grp = grp.sort_values("x")
        ax.plot(grp["x"], grp["y"], marker="o", label=group)
        if metric == "ripley_K" and not envelope_drawn and grp[["y_lo", "y_hi"]].notna().all(axis=None):
            ax.fill_between(grp["x"].astype(float), grp["y_lo"].astype(float), grp["y_hi"].astype(float), color="#808080", alpha=0.20, linewidth=0, label="null envelope")
            envelope_drawn = True
    if metric in {"kernel_eigen_scree", "participation_ratio_scree"}:
        finite_x = pd.to_numeric(sub["x"], errors="coerce").dropna()
        if not finite_x.empty:
            ax.set_xlim(1, min(30, float(finite_x.max())))
    ax.set_title(title)
    xlabel, ylabel = axis_labels.get(metric, ("curve x", "curve y"))
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    fig.tight_layout()
    _save_fig(fig, out_base)

def _plot_coverage_scatter(tests, out_base, title):
    sub = _metric_rows(tests, "coverage", "coverage_geometric")
    if sub.empty:
        return _empty_panel(out_base, title, "No coverage rows.")
    fig, ax = plt.subplots(figsize=(6, 5))
    for _, row in sub.iterrows():
        ax.scatter(row["ai_value"], row["effect_size"], s=80, color=GROUP_COLORS.get(row["comparison"], "#808080"))
        ax.text(row["ai_value"], row["effect_size"], row["comparison"].replace("human_vs_", ""), fontsize=8)
    ax.axvline(1.0, color="#404040", linestyle="--", linewidth=1)
    ax.set_xlabel("coverage: fraction of human review span reached")
    ax.set_ylabel("paired Cliff's delta")
    ax.set_title(title)
    fig.tight_layout()
    _save_fig(fig, out_base)

def _plot_umap(condition, text_version, out_base, title):
    prep = PROJECT_ROOT / "data" / "prepared" / condition / "reviews" / text_version
    coords_path = prep / "review_umap2d.npy"
    master_path = prep / "review_master.csv"
    if not coords_path.exists() or not master_path.exists():
        return _empty_panel(out_base, title, "Missing review UMAP prep artifacts.")
    coords = np.load(coords_path)
    master = pd.read_csv(master_path)
    fig, ax = plt.subplots(figsize=(6, 5))
    for label, grp in master.assign(_x=coords[:, 0], _y=coords[:, 1]).groupby("source_group"):
        ax.scatter(grp["_x"], grp["_y"], s=16, alpha=0.65, label=label)
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.set_title(title + "\nillustration only; metrics use full embedding space")
    ax.legend(fontsize=8, loc="best")
    fig.tight_layout()
    _save_fig(fig, out_base)

def emit_required_figures(condition, text_version):
    tables_dir = PROJECT_ROOT / "results" / "tables" / condition / "reviews" / text_version
    figures_dir = PROJECT_ROOT / "results" / "figures" / condition / "reviews" / text_version
    tests = pd.read_csv(tables_dir / "facet_diversity_tests.csv")
    paired = pd.read_csv(tables_dir / "facet_review_paired_long.csv")
    curves = pd.read_parquet(tables_dir / "facet_diversity_curves.parquet")
    for stem, title_base, kind, facet, metric in REQUIRED_REVIEW_FIGURES:
        title = f"{title_base} · {condition} · reviews/{text_version}"
        out_base = figures_dir / stem
        if kind == "paired_box":
            _plot_paired_box(paired, out_base, title, facet, metric)
        elif kind == "paired_hist":
            _plot_paired_hist(paired, out_base, title, facet, metric)
        elif kind == "effect":
            _plot_effect(tests, out_base, title, facet, metric)
        elif kind == "convergent_box":
            _plot_convergent_box(paired, out_base, title)
        elif kind == "paired_profile":
            _plot_paired_profile(paired, out_base, title, facet, metric)
        elif kind == "curve_required":
            _plot_curve_required(curves, out_base, title, facet, metric)
        elif kind == "coverage_scatter":
            _plot_coverage_scatter(tests, out_base, title)
        elif kind == "umap":
            _plot_umap(condition, text_version, out_base, title)
    for facet, metric in [("spread", "mean_pairwise"), ("richness", "vendi"), ("coverage", "coverage_geometric"), ("dimensionality", "participation_ratio"), ("evenness", "ripley_excess")]:
        _plot_paired_profile(
            paired,
            figures_dir / f"{facet}_{metric}_paired_slope",
            f"{facet.title()} — {metric} paired slopes · {condition} · reviews/{text_version}",
            facet,
            metric,
        )

if WRITE_FIGURES:
    for condition in CONFIG["conditions"]:
        for text_version in CONFIG["text_versions"]:
            emit_required_figures(condition, text_version)
print("Review facet notebook complete.")


Review facet notebook complete.
